# A3-06 · Single-model D3 holdout (selected after development CV)

**A separate, lightweight evaluation of ONE frozen configuration, not completion of the full A3 holdout protocol.**
Model: `exp8a_xu_dual_gcn_surface` / `plv__psd_de_strength__abl_anatomical_both`, wMNE, mode B, lobes12, dense
(`configs/final/amendments/a3_single_model.yaml`). It was chosen by the author after development CV to save computation;
the other 115 frozen configurations are **not** evaluated here, and the 13-participant holdout was inspected earlier, so the
result is descriptive only. The A3 master freeze, both frozen records, the 116-unit inventory and all development results
are only read.

It verifies that the exact unit is in the frozen A2 record and that its development result is intact (the other results
are not rescanned), restores the numerical environment of the A2 results and checks it against this unit's development
environment, then fits the unit once on the 52 development participants and evaluates the 13 holdout participants **at the
A2 pin** (`7fa470c`, unchanged model and protocol). Resumable: a completed unit is not recomputed. GPU runtime (T4).

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook again
and *Run all* again: finished work is kept on Drive. Everything A3 writes goes to `<outputs>/FINAL_v1/amendments/A3_final/`
(plus, once, the two frozen records created by the freeze); no completed result, pin, QC decision or store is changed.

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the A3 code, installs the requirements. =====
UPDATE_CODE = False   # leave False. True only to move the A3 code pin (<outputs>/FINAL_v1/amendments/A3_final/CODE_COMMIT.txt)
BRANCH = 'claude/a3-final-analysis'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])   # full history: the pins are ancestors
os.chdir('/content/repo')
_run([sys.executable, 'scripts/a3_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')

from adgraph.colab import sh as _sh

def sh(cmd, cwd='/content/repo'):  # every script runs with the file backend
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('/content/repo/configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
A3 = FINAL + '/amendments/A3_final'
print('A3 outputs on Drive:', A3)

## 1. Numerical environment (restored to the one recorded by this unit's development result)

In [ ]:
ACCEPT_ENVIRONMENT_DIFFERENCE = False   # leave False: a difference that cannot be removed must stop here
from adgraph import numerical_env
from adgraph.a3 import single
# The environment recorded by THIS unit's development result (one small file; the ~320 other results are not scanned).
REFERENCE = single.dev_libraries(OUT)
PY = numerical_env.ensure_interpreter(REFERENCE, repo='/content/repo', base='/content',
                                      accept_difference=ACCEPT_ENVIRONMENT_DIFFERENCE, run=sh)
NOW = numerical_env.interpreter_versions(PY, REFERENCE)
DIFF = numerical_env.differences(REFERENCE, NOW)
assert ACCEPT_ENVIRONMENT_DIFFERENCE or not DIFF, f'numerical environment differs from the development environment of this unit: {DIFF}'
print('the holdout runs with', PY, NOW)

## 2. Membership in the frozen inventory and integrity checks (read only)

In [ ]:
sh("python scripts/a3.py single-verify")

## 3. Holdout of this one unit (writes the selection manifest once; resumable)

In [ ]:
sh(f"python scripts/a3.py single-holdout --python {PY} --repo /content/repo --base /content")

## 4. Report of this one model (does not need the other frozen units)

In [ ]:
sh("python scripts/a3.py single-report")

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
def show_latest(kind):
    d = latest(A3 + f'/reports/{kind}_*')
    print(d)
    return d
d = latest(A3 + '/single_model/report_*')
print(d)
print(open(d + '/single_model_holdout_report.md').read())
display(pd.read_csv(d + '/predictions_participants.csv'))
for png in sorted(glob.glob(d + '/*.png')):
    display(Image(filename=png))